In [2]:
import pandas as pd
df = pd.read_csv("ai_ready_exercise.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'ai_ready_exercise.csv'

In [ ]:
print(df.columns.tolist())

['name', 'description', 'category', 'muscle', 'equipment', 'difficulty', 'rating', 'rating_description', 'image_url']


In [ ]:
df["rating"] = df["rating"].round(1)

In [ ]:
def get_muscle_map(name: str, muscle: str = None):
    """
    Fixed version - No more unwanted 'arms' in legs/glutes exercises.
    """
    if name is None:
        name = ""
    name = str(name).lower().strip()

    if muscle is None:
        muscle = ""
    muscle = str(muscle).lower().strip()

    def has(*keywords):
        return any(k in name for k in keywords)

    is_upper = has("upper", "incline", "high")
    is_lower = has("lower", "decline", "bottom", "reverse")

    primary = None
    secondary = []

    # ==================== PRIMARY MUSCLE ====================
    if has("hip thrust", "glute bridge", "kas glute", "frog pump", "thrust", 
           "kickback", "donkey kick", "cable kickback", "hip extension"):
        primary = "glute_maximus_upper" if is_upper else "glute_maximus"

    elif has("leg extension", "sissy squat"):
        primary = "legs_quads"

    elif has("leg curl", "hamstring curl", "nordic curl", "good morning"):
        primary = "legs_hamstrings"

    elif has("bench press", "chest press", "dip", "push up", "fly"):
        if is_upper or has("incline"):
            primary = "chest_pectorals_upper"
        elif is_lower or has("decline"):
            primary = "chest_pectorals_lower"
        else:
            primary = "chest_pectorals"

    elif has("crunch", "sit up", "cable crunch"):
        primary = "rectus_abdominis_upper" if is_upper else "rectus_abdominis_lower" if is_lower else "rectus_abdominis"

    elif has("leg raise", "hanging leg raise", "reverse crunch", "v-up"):
        primary = "rectus_abdominis_lower"

    elif has("lateral raise", "side raise"):
        primary = "shoulders_lateral"

    elif has("rear delt", "reverse fly"):
        primary = "shoulders_rear"

    elif has("shoulder press", "overhead press", "military press"):
        primary = "shoulders_anterior"

    elif has("pull up", "chin up", "lat pulldown"):
        primary = "back_lats"

    elif has("row", "seated row", "barbell row"):
        primary = "back_rhomboids"

    elif has("bicep curl", "hammer curl"):
        primary = "arms_biceps"

    elif has("tricep", "pushdown", "skull crusher"):
        primary = "arms_triceps"

    elif has("calf raise"):
        primary = "calves_gastrocnemius" if has("standing", "donkey") else "calves_soleus"

    # ==================== FALLBACK FROM 'muscle' COLUMN ====================
    if primary is None and muscle:
        muscle_map = {
            "glutes": "glute_maximus",
            "chest": "chest_pectorals",
            "core": "rectus_abdominis",
            "arms": "arms_biceps",
            "legs": "legs_quads",
            "middle back": "back_rhomboids",
            "back": "back_lats",
            "shoulders": "shoulders_anterior",
            "calves": "calves_gastrocnemius"
        }
        primary = muscle_map.get(muscle)

    # Final fallback
    if primary is None:
        if has("squat", "lunge", "bulgarian", "step up"):
            primary = "legs_quads"
        elif has("deadlift", "rdl", "romanian"):
            primary = "legs_hamstrings"
        elif has("plank", "ab rollout", "ab wheel"):
            primary = "transverse_core"
        elif has("russian twist", "woodchopper", "side plank"):
            primary = "obliques"
        else:
            primary = "full_body"

    # ==================== FIXED SECONDARY MUSCLES ====================
    # Only add secondary muscles when it makes sense
    if has("squat", "lunge", "leg press", "bulgarian", "step up"):
        secondary.extend(["glute_maximus", "legs_quads"])

    elif has("deadlift", "rdl", "romanian"):
        secondary.extend(["glute_maximus", "legs_hamstrings"])

    elif has("bench press", "chest press", "dip", "push up", "fly"):
        secondary.extend(["shoulders_anterior", "arms_triceps"])

    elif has("pull up", "chin up", "lat pulldown"):
        secondary.append("arms_biceps")

    elif has("plank", "hollow", "ab rollout", "ab wheel"):
        secondary.append("transverse_core")

    # Clean up secondary list
    secondary = list(dict.fromkeys(secondary))
    if primary in secondary:
        secondary.remove(primary)

    # ====================== IMAGE MAP ======================
    image_map = {
        "glute_maximus": "https://mdhealth.com.au/wp-content/uploads/2014/07/gluteus-maximus-1000x675.jpg",
        "glute_maximus_upper": "https://mdhealth.com.au/wp-content/uploads/2014/07/gluteus-maximus-1000x675.jpg",
        "rectus_abdominis": "https://www.madscientistofmuscle.com/1-exercises/1-muscle-anatomy/graphics/rectus-abdominis.jpg",
        "rectus_abdominis_upper": "https://www.madscientistofmuscle.com/1-exercises/1-muscle-anatomy/graphics/rectus-abdominis.jpg",
        "rectus_abdominis_lower": "https://www.madscientistofmuscle.com/1-exercises/1-muscle-anatomy/graphics/rectus-abdominis.jpg",
        "transverse_core": "https://kinxlearning.com/cdn/shop/files/muscle-61_500x.jpg?v=1613518563",
        "obliques": "https://cdn.shopify.com/s/files/1/1214/5580/files/Muscle_Group_Abs.jpg?v=1601051622",

        "chest_pectorals": "https://cdn.shopify.com/s/files/1/1214/5580/files/Muscle_Group_Chest.jpg?v=1601050935",
        "chest_pectorals_upper": "https://cdn.shopify.com/s/files/1/1214/5580/files/Muscle_Group_Chest.jpg?v=1601050935",
        "chest_pectorals_lower": "https://cdn.shopify.com/s/files/1/1214/5580/files/Muscle_Group_Chest.jpg?v=1601050935",

        "shoulders_lateral": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1e/Deltoid_muscle.PNG/500px-Deltoid_muscle.PNG",
        "shoulders_rear": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1e/Deltoid_muscle.PNG/500px-Deltoid_muscle.PNG",
        "shoulders_anterior": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1e/Deltoid_muscle.PNG/500px-Deltoid_muscle.PNG",

        "back_lats": "https://blog.warmbody-coldmind.com/wp-content/uploads/2024/05/training-barbell-lat-workout-latissimus-dorsi-anatomy.jpg",
        "back_rhomboids": "https://blog.infs.com/wp-content/uploads/2025/03/129458131_m.jpg",

        "arms_biceps": "https://t3.ftcdn.net/jpg/05/04/89/18/240_F_504891829_C8C4uvSyiarKli1wYpnDmNvkqiWsE2KN.jpg",
        "arms_triceps": "https://barbend.com/wp-content/uploads/2019/04/Tricep-Muscle-Anatomy-1024x661.jpg",

        "legs_quads": "https://www.shutterstock.com/image-illustration/quadriceps-male-muscles-anatomy-muscle-260nw-489727177.jpg",
        "legs_hamstrings": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcS_OWv94jxxO7g7rUVzmTJelOJFvjrh579tqA&s",

        "calves_gastrocnemius": "https://cdn.shopify.com/s/files/1/1214/5580/files/Muscle_Group_Calves.jpg?v=1601051316",
        "calves_soleus": "https://cdn.shopify.com/s/files/1/1214/5580/files/Muscle_Group_Calves.jpg?v=1601051316",

        "full_body": None,
    }

    image_url = image_map.get(primary)
    secondary_images = [image_map.get(m) for m in secondary if image_map.get(m)]

    return {
        "primary": primary,
        "secondary": secondary,
        "image_url": image_url,
        "secondary_images": secondary_images
    }

In [ ]:
df["primary_muscle"]   = df["name"].apply(lambda x: get_muscle_map(x)["primary"])
df["muscle_region"]    = df["name"].apply(lambda x: ", ".join([get_muscle_map(x)["primary"]] + get_muscle_map(x)["secondary"]))
df["image_url"]        = df["name"].apply(lambda x: get_muscle_map(x)["image_url"])
df["secondary_images"] = df["name"].apply(lambda x: get_muscle_map(x)["secondary_images"])

In [ ]:
import pandas as pd

# Load your existing dataset
df = pd.read_csv("ai_ready_exercises.csv")

# ============================
# Function to add exercises
# ============================
def add_glute_medius_minimus_exercises(df):

    new_exercises = [
        {
            "name": "Side-Lying Hip Abduction",
            "description": "Lie on your side with legs straight. Lift the top leg upward while keeping it straight and in line with your body.",
            "category": "Lower Body",
            "muscle": "glutes",
            "equipment": "Bodyweight",
            "difficulty": "Beginner",
            "rating": 4.5,
            "rating_description": "Excellent isolation exercise for glute medius and minimus",
            "primary_muscle": "glute_medius",
            "muscle_region": "glute_medius, glute_minimus",
            "image_url": "https://nakoafit.com/wp-content/uploads/2023/10/gluteus-medius.jpg",
            "secondary_images": ""
        },
        {
            "name": "Clamshell",
            "description": "Lie on your side with knees bent at 45 degrees and feet together. Lift the top knee while keeping feet touching.",
            "category": "Lower Body",
            "muscle": "glutes",
            "equipment": "Bodyweight / Resistance Band",
            "difficulty": "Beginner",
            "rating": 4.4,
            "rating_description": "Classic activation exercise targeting glute medius and minimus",
            "primary_muscle": "glute_medius",
            "muscle_region": "glute_medius, glute_minimus, transverse_core",
            "image_url": "https://nakoafit.com/wp-content/uploads/2023/10/gluteus-medius.jpg",
            "secondary_images": ""
        },
        {
            "name": "Fire Hydrant",
            "description": "On all fours, lift one bent knee out to the side while keeping your core stable.",
            "category": "Lower Body",
            "muscle": "glutes",
            "equipment": "Bodyweight",
            "difficulty": "Beginner",
            "rating": 4.3,
            "rating_description": "Great for building side glute strength and hip stability",
            "primary_muscle": "glute_medius",
            "muscle_region": "glute_medius, glute_minimus, transverse_core",
            "image_url": "https://nakoafit.com/wp-content/uploads/2023/10/gluteus-medius.jpg",
            "secondary_images": ""
        },
        {
            "name": "Lateral Band Walk",
            "description": "Place resistance band above knees. Take small side steps in a slight squat position.",
            "category": "Lower Body",
            "muscle": "glutes",
            "equipment": "Resistance Band",
            "difficulty": "Intermediate",
            "rating": 4.6,
            "rating_description": "Highly effective for glute medius and minimus activation",
            "primary_muscle": "glute_medius",
            "muscle_region": "glute_medius, glute_minimus, legs_adductors",
            "image_url": "https://nakoafit.com/wp-content/uploads/2023/10/gluteus-medius.jpg",
            "secondary_images": ""
        },
        {
            "name": "Standing Cable Hip Abduction",
            "description": "Attach ankle strap to low cable pulley. Abduct leg out to the side against resistance.",
            "category": "Lower Body",
            "muscle": "glutes",
            "equipment": "Cable Machine",
            "difficulty": "Intermediate",
            "rating": 4.5,
            "rating_description": "Controlled resistance for glute medius and minimus",
            "primary_muscle": "glute_medius",
            "muscle_region": "glute_medius, glute_minimus",
            "image_url": "https://nakoafit.com/wp-content/uploads/2023/10/gluteus-medius.jpg",
            "secondary_images": ""
        }
    ]

    new_df = pd.DataFrame(new_exercises)

    # ============================
    # 🔥 ALIGN COLUMNS (IMPORTANT)
    # ============================

    # Add missing columns to original df
    for col in new_df.columns:
        if col not in df.columns:
            df[col] = None

    # Add missing columns to new_df
    for col in df.columns:
        if col not in new_df.columns:
            new_df[col] = None

    # Ensure same column order
    new_df = new_df[df.columns]

    # ============================
    # CONCAT
    # ============================
    df = pd.concat([df, new_df], ignore_index=True)

    # Optional: remove duplicates by name
    df = df.drop_duplicates(subset=["name"], keep="first")

    # Sort
    df = df.sort_values(by="name").reset_index(drop=True)

    return df


# ============================
# RUN IT
# ============================
df = add_glute_medius_minimus_exercises(df)

# Save
df.to_csv("ai_ready_exercises_updated.csv", index=False)

print("✅ Done — exercises added cleanly!")

✅ Done — exercises added cleanly!


In [ ]:
df

In [ ]:
df.to_csv("ai_ready_exercise.csv", index=False)

In [58]:
import pandas as pd

df = pd.read_csv("exercise_neck_fixed.csv")

# -----------------------------
# FILTER: Glutes with NULL rating
# -----------------------------
glutes_null_rating = df[
    (df["BodyPart"] == "Glutes") &
    (df["Rating"].isna())
]

# -----------------------------
# OUTPUT
# -----------------------------
print("\n🚨 GLUTE EXERCISES WITH NULL RATING:")
print(glutes_null_rating[["Title", "Type", "BodyPart"]].to_string(index=False))

print("\n📊 SUMMARY")
print("Total Glutes:", len(df[df["BodyPart"] == "Glutes"]))
print("Null Ratings in Glutes:", len(glutes_null_rating))


🚨 GLUTE EXERCISES WITH NULL RATING:
                                      Title     Type BodyPart
                      KV Barbell Hip Thrust Strength   Glutes
         Dumbbell straight-legged hip raise Strength   Glutes
                      Glute bridge step-out Strength   Glutes
Natural Glute Ham Raise with Stability Ball Strength   Glutes
               MetaBurn Tabletop Hip Thrust Strength   Glutes
       Holman Froggy Push-Up to Donkey Kick Strength   Glutes
                         Band good morning- Strength   Glutes
                      UN Cable Pull-Through Strength   Glutes
           Holman Bear Crawl to Donkey Kick Strength   Glutes

📊 SUMMARY
Total Glutes: 56
Null Ratings in Glutes: 9


In [62]:
import pandas as pd
import numpy as np

df = pd.read_csv("exercise_final.csv")

glutes = df[df["BodyPart"] == "Glutes"]

# -----------------------------
# 1. CHECK 1: NULL RATINGS
# -----------------------------
null_ratings = glutes[glutes["Rating"].isna()]

# -----------------------------
# 2. CHECK 2: OUT OF RANGE RATINGS
# -----------------------------
out_of_range = glutes[
    (glutes["Rating"] < 0) | (glutes["Rating"] > 10)
]

# -----------------------------
# 3. CHECK 3: SUSPICIOUS NON-GLUTE PATTERNS
# -----------------------------
def is_suspicious(title):
    t = str(title).lower()

    # not glute dominant patterns
    if any(x in t for x in [
        "neck", "triceps", "biceps",
        "chest", "shoulder"
    ]):
        return True

    # mobility-only movements (low glute relevance)
    if any(x in t for x in [
        "stretch", "smr", "mobility", "yoga"
    ]):
        return True

    return False


suspicious = glutes[glutes["Title"].apply(is_suspicious)]

# -----------------------------
# 4. CHECK 4: LOW QUALITY GLUTE SCORE (optional flag)
# -----------------------------
low_quality = glutes[glutes["Rating"] < 3.5]

# -----------------------------
# 5. SUMMARY METRICS
# -----------------------------
total = len(glutes)
issues = len(null_ratings) + len(out_of_range) + len(suspicious)

reliability = round(((total - issues) / total) * 100, 2)

# -----------------------------
# 6. REPORT
# -----------------------------
print("\n📊 GLUTES DATASET VALIDATION REPORT")

print("\n✔ Total Glutes:", total)

print("\n🚨 Null Ratings:", len(null_ratings))
if len(null_ratings) > 0:
    print(null_ratings[["Title", "Rating"]].to_string(index=False))

print("\n🚨 Out of Range Ratings:", len(out_of_range))

print("\n⚠ Suspicious Entries:", len(suspicious))
if len(suspicious) > 0:
    print(suspicious[["Title", "Rating"]].to_string(index=False))

print("\n⚠ Low Quality (<3.5):", len(low_quality))

print("\n📈 FINAL RELIABILITY SCORE:", reliability, "%")

# -----------------------------
# 7. FINAL VERDICT
# -----------------------------
if reliability >= 95:
    print("\n🟢 STATUS: READY FOR USE")
elif reliability >= 85:
    print("\n🟡 STATUS: USABLE WITH CAUTION")
else:
    print("\n🔴 STATUS: NEEDS FIXES")


📊 GLUTES DATASET VALIDATION REPORT

✔ Total Glutes: 56

🚨 Null Ratings: 0

🚨 Out of Range Ratings: 0

⚠ Suspicious Entries: 0

⚠ Low Quality (<3.5): 1

📈 FINAL RELIABILITY SCORE: 100.0 %

🟢 STATUS: READY FOR USE


In [70]:
import pandas as pd

df = pd.read_csv("exercise_level_fixed.csv")

mask = df["BodyPart"] == "Glutes"

# -----------------------------
# STRICT GLUTE LEVEL ENGINE
# -----------------------------
def assign_level(title):
    t = str(title).lower()

    # 🔴 EXPERT (high load / explosive / unstable / advanced anatomy)
    if any(x in t for x in [
        "barbell",
        "glute ham raise",
        "smith",
        "box jump",
        "plyometric",
        "thruster",
        "single-leg cable",
        "suspended"
    ]):
        return "Expert"

    # 🟡 INTERMEDIATE (loaded + coordination)
    if any(x in t for x in [
        "pull-through",
        "good morning",
        "step-up",
        "lunge",
        "single-leg",
        "band",
        "leg press",
        "hip lift"
    ]):
        return "Intermediate"

    # 🟢 BEGINNER (activation + isolation only)
    if any(x in t for x in [
        "glute bridge",
        "hip extension",
        "donkey kick",
        "kickback",
        "side-lying",
        "monster walk",
        "standing hip"
    ]):
        return "Beginner"

    # fallback (safer default)
    return "Intermediate"


# -----------------------------
# OVERWRITE LEVELS (GLUTES ONLY)
# -----------------------------
df.loc[mask, "Level"] = df.loc[mask, "Title"].apply(assign_level)

# -----------------------------
# CHECK DISTRIBUTION
# -----------------------------
print("\n📊 FINAL CLEAN GLUTE LEVELS:")
print(df[df["BodyPart"] == "Glutes"]["Level"].value_counts())

# -----------------------------
# SAMPLE CHECK
# -----------------------------
print("\n🧪 SAMPLE CLEANED DATA:")
print(df[df["BodyPart"] == "Glutes"][["Title", "Level", "Rating"]].head(20))

# -----------------------------
# SAVE CLEAN VERSION
# -----------------------------
df.to_csv("exercise_FINAL_CLEAN.csv", index=False)

print("\n✅ Saved: exercise_FINAL_CLEAN.csv")


📊 FINAL CLEAN GLUTE LEVELS:
Level
Intermediate    32
Expert          13
Beginner        11
Name: count, dtype: int64

🧪 SAMPLE CLEANED DATA:
                                            Title         Level  Rating
19                          KV Barbell Hip Thrust        Expert     9.0
57             Dumbbell straight-legged hip raise  Intermediate     4.9
277                         Glute bridge step-out      Beginner     5.1
333   Natural Glute Ham Raise with Stability Ball        Expert     4.1
595                  MetaBurn Tabletop Hip Thrust  Intermediate     9.1
670                                Thigh abductor  Intermediate     8.2
1013         Holman Froggy Push-Up to Donkey Kick      Beginner     4.5
1208                     Hip Extension with Bands  Intermediate     7.2
1209                           Hip Lift with Band  Intermediate     6.0
1210                              HM Monster Walk      Beginner     4.6
1211                      UNS Banded Glute Bridge  Intermediate   

In [66]:
import pandas as pd

df = pd.read_csv("exercise_final.csv")

glutes_mask = df["BodyPart"] == "Glutes"

# -----------------------------
# STRICT LEVEL CLASSIFIER
# -----------------------------
def assign_level(title):
    t = str(title).lower()

    # 🔴 EXPERT (high load / unstable / complex)
    if any(x in t for x in [
        "barbell",
        "glute ham raise",
        "box jump",
        "plyometric",
        "thruster",
        "single-leg cable",
        "smith",
        "suspended"
    ]):
        return "Expert"

    # 🟡 INTERMEDIATE (loaded + coordination)
    if any(x in t for x in [
        "pull-through",
        "good morning",
        "step-up",
        "lunge",
        "single-leg",
        "band",
        "leg press"
    ]):
        return "Intermediate"

    # 🟢 BEGINNER (ONLY activation + isolation)
    if any(x in t for x in [
        "glute bridge",
        "hip extension",
        "donkey kick",
        "kickback",
        "side-lying",
        "monster walk"
    ]):
        return "Beginner"

    # fallback (important!)
    return "Intermediate"


# -----------------------------
# APPLY ONLY TO GLOUTES
# -----------------------------
df.loc[glutes_mask, "Level"] = df.loc[glutes_mask, "Title"].apply(assign_level)

# -----------------------------
# CHECK DISTRIBUTION
# -----------------------------
print("\n📊 BALANCED LEVEL DISTRIBUTION (GLOUTES):")
print(df[df["BodyPart"] == "Glutes"]["Level"].value_counts())

# -----------------------------
# SAVE
# -----------------------------
df.to_csv("exercise_level_fixed.csv", index=False)

print("\n✅ Saved: exercise_level_fixed.csv")


📊 BALANCED LEVEL DISTRIBUTION (GLOUTES):
Level
Intermediate    32
Expert          13
Beginner        11
Name: count, dtype: int64

✅ Saved: exercise_level_fixed.csv


In [86]:
import pandas as pd

df = pd.read_csv("exercise_FINAL_CLEAN.csv")

glutes = df[df["BodyPart"] == "Glutes"].copy()

# -----------------------------
# CLEAN EQUIPMENT TEXT (IMPORTANT)
# -----------------------------
glutes["Equipment_clean"] = (
    glutes["Equipment"]
    .fillna("Unknown")
    .str.lower()
    .str.strip()
)

# -----------------------------
# EQUIPMENT DISTRIBUTION
# -----------------------------
equipment_counts = glutes["Equipment_clean"].value_counts()

print("\n📊 EQUIPMENT DISTRIBUTION (GLOUTES):\n")
print(equipment_counts)

# -----------------------------
# SHOW SAMPLE PER EQUIPMENT TYPE
# -----------------------------
print("\n🧪 SAMPLE EXAMPLES PER EQUIPMENT:\n")

for eq in equipment_counts.index:
    sample = glutes[glutes["Equipment_clean"] == eq][["Title", "Equipment_clean"]].head(3)
    print(f"\n--- {eq.upper()} ---")
    print(sample.to_string(index=False))

# -----------------------------
# FLAG POTENTIAL ISSUES
# -----------------------------
print("\n🚨 POTENTIAL DATA ISSUES:\n")

# very common problem: messy or mixed labels
bad_labels = glutes[
    glutes["Equipment_clean"].str.contains("unknown|nan|none|body weight|bodyweight|mixed", na=False)
]

print("Unclear equipment entries:", len(bad_labels))
print(bad_labels[["Title", "Equipment_clean"]].head(20))


📊 EQUIPMENT DISTRIBUTION (GLOUTES):

Equipment_clean
body only        30
bands             6
machine           4
dumbbell          3
barbell           3
cable             3
exercise ball     2
plyometrics       2
kettlebells       1
medicine ball     1
unknown           1
Name: count, dtype: int64

🧪 SAMPLE EXAMPLES PER EQUIPMENT:


--- BODY ONLY ---
                                      Title Equipment_clean
                      Glute bridge step-out       body only
Natural Glute Ham Raise with Stability Ball       body only
               MetaBurn Tabletop Hip Thrust       body only

--- BANDS ---
                   Title Equipment_clean
Hip Extension with Bands           bands
      Hip Lift with Band           bands
         HM Monster Walk           bands

--- MACHINE ---
                  Title Equipment_clean
         Thigh abductor         machine
30 Legs Glute-Ham Raise         machine
        Glute Ham Raise         machine

--- DUMBBELL ---
                                

In [156]:
import pandas as pd

# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv("exercise_FINAL_CLEAN.csv")

# Filter glutes only
glutes = df[df["BodyPart"] == "Glutes"].copy()

# Fill missing ratings
glutes["Rating"] = glutes["Rating"].fillna(0)

# Normalize equipment text
glutes["Equipment"] = glutes["Equipment"].str.lower()


# =========================================================
# STIMULUS CLASSIFIER
# =========================================================
def get_stimulus(title):

    t = str(title).lower()

    # Mechanical tension
    if "hip thrust" in t or "glute bridge" in t:
        return "tension"

    # Stretch bias
    if (
        "extension" in t
        or "good morning" in t
        or "ham" in t
        or "rdl" in t
    ):
        return "stretch"

    # Isolation / activation
    if (
        "donkey" in t
        or "kick" in t
        or "monster" in t
        or "abductor" in t
    ):
        return "activation"

    # Stability
    if (
        "step" in t
        or "single" in t
        or "balance" in t
    ):
        return "stability"

    # Metabolic / burn
    if (
        "band" in t
        or "duck" in t
        or "pulse" in t
    ):
        return "metabolic"

    return "other"


glutes["stimulus"] = glutes["Title"].apply(get_stimulus)


# =========================================================
# MOVEMENT PATTERN DETECTION
# =========================================================
def get_pattern(title):

    t = str(title).lower()

    if "hip thrust" in t:
        return "hip_thrust"

    if "glute bridge" in t:
        return "glute_bridge"

    if "ham" in t:
        return "hamstring"

    if "kick" in t:
        return "kickback"

    if "step" in t:
        return "step"

    if "abductor" in t:
        return "abduction"

    return "other"


glutes["pattern"] = glutes["Title"].apply(get_pattern)


# =========================================================
# GOAL WEIGHTS
# =========================================================
def goal_weights(goal):

    if goal == "bulking":
        return {
            "barbell": 4,
            "machine": 3,
            "dumbbell": 2,
            "cable": 2,
            "bands": 0,
            "body only": -2
        }

    if goal == "cutting":
        return {
            "bands": 3,
            "body only": 3,
            "cable": 2,
            "exercise ball": 2,
            "machine": 1
        }

    if goal == "recomp":
        return {
            "barbell": 2,
            "machine": 2,
            "dumbbell": 2,
            "cable": 2,
            "bands": 1,
            "body only": 0
        }

    return {}


# =========================================================
# SCORE SYSTEM
# =========================================================
def score_row(row, goal):

    score = row["Rating"]

    eq = row["Equipment"]
    stim = row["stimulus"]

    weights = goal_weights(goal)

    # Equipment scoring
    score += weights.get(eq, 0)

    # Stimulus bonuses
    if stim == "tension":
        score += 1.5

    if stim == "stretch":
        score += 1

    if stim == "activation":
        score += 0.5

    return score


# =========================================================
# PREP DATA
# =========================================================
def prepare_data(goal):

    data = glutes.copy()

    data["score"] = data.apply(
        lambda r: score_row(r, goal),
        axis=1
    )

    # Sort highest first
    data = data.sort_values("score", ascending=False)

    return data


# =========================================================
# DAY STRUCTURE
# =========================================================
DAY_STRUCTURE = {
    "activation": 1,
    "tension": 2,
    "stretch": 1,
    "stability": 1,
    "metabolic": 1
}


# =========================================================
# LEVEL MULTIPLIERS
# =========================================================
LEVEL_MULTIPLIER = {
    "Beginner": 0.6,
    "Intermediate": 1.0,
    "Expert": 1.6
}


# =========================================================
# LOAD PRESCRIPTION
# =========================================================
def prescribe_load(row, goal, level):

    stim = row["stimulus"]
    eq = row["Equipment"]
    rating = row["Rating"]

    # -------------------------
    # REP RANGES
    # -------------------------
    if stim == "tension":
        reps = "6-10"

    elif stim == "stretch":
        reps = "8-12"

    elif stim == "activation":
        reps = "12-20"

    elif stim == "stability":
        reps = "10-15"

    else:
        reps = "15-25"

    # -------------------------
    # SETS
    # -------------------------
    if goal == "bulking":
        sets = 4 if stim == "tension" else 3

    elif goal == "cutting":
        sets = 3

    else:  # recomp
        sets = 4 if stim == "tension" else 3

    # -------------------------
    # EQUIPMENT LOAD FACTORS
    # -------------------------
    equipment_factor = {
        "barbell": 1.0,
        "machine": 0.9,
        "dumbbell": 0.7,
        "cable": 0.6,
        "bands": 0.3,
        "body only": 0.0,
        "exercise ball": 0.2,
        "medicine ball": 0.3
    }

    factor = equipment_factor.get(eq, 0.5)

    level_mult = LEVEL_MULTIPLIER.get(level, 1.0)

    # -------------------------
    # REALISTIC START WEIGHT
    # -------------------------
    start_weight = round(
        rating * 6 * factor * level_mult,
        1
    )

    # Bodyweight exercises
    if eq == "body only":
        start_weight = 0

    return sets, reps, start_weight


# =========================================================
# PICK EXERCISES
# =========================================================
def pick(df, stim, n, used_titles, used_patterns, goal):

    pool = df[
        (df["stimulus"] == stim) &
        (~df["Title"].isin(used_titles)) &
        (~df["pattern"].isin(used_patterns))
    ]

    # Bulking prioritizes strongest movements
    if goal == "bulking":
        pool = pool.sort_values("Rating", ascending=False)

    return pool.head(n)


# =========================================================
# BUILD TRAINING DAY
# =========================================================
def build_day(df, goal, level):

    used_titles = set()
    used_patterns = set()

    day = []

    for stim, count in DAY_STRUCTURE.items():

        selected = pick(
            df,
            stim,
            count,
            used_titles,
            used_patterns,
            goal
        )

        # Fallback
        if len(selected) < count:

            fallback = df[
                (df["stimulus"] == stim) &
                (~df["Title"].isin(used_titles))
            ].head(count - len(selected))

            selected = pd.concat([selected, fallback])

        selected = selected.drop_duplicates(subset=["Title"]).copy()

        # Add programming variables
        selected[["sets", "reps", "start_weight"]] = selected.apply(
            lambda r: pd.Series(
                prescribe_load(r, goal, level)
            ),
            axis=1
        )

        used_titles.update(selected["Title"].tolist())
        used_patterns.update(selected["pattern"].tolist())

        day.append(selected)

    return pd.concat(day)


# =========================================================
# GENERATE PROGRAM
# =========================================================
def generate_plan(goal="recomp", level="Intermediate"):

    data = prepare_data(goal)

    # -------------------------
    # LEVEL FILTERS
    # -------------------------
    if level == "Beginner":
        data = data[data["Rating"] <= 7.5]

    elif level == "Expert":
        data = data.sort_values("Rating", ascending=False)

    # -------------------------
    # BUILD DAYS
    # -------------------------
    day1 = build_day(data, goal, level)

    remaining = data[
        ~data["Title"].isin(day1["Title"])
    ]

    day2 = build_day(remaining, goal, level)

    return day1, day2


# =========================================================
# USER INPUT
# =========================================================
goal = input(
    "Goal (cutting / bulking / recomp): "
).strip().lower()

level = input(
    "Level (Beginner / Intermediate / Expert): "
).strip().title()


# =========================================================
# GENERATE
# =========================================================
day1, day2 = generate_plan(goal, level)


# =========================================================
# OUTPUT
# =========================================================
print("\n🟢 GLUTE AI — FINAL DETERMINISTIC PROGRAM")
print(f"Goal: {goal}")
print(f"Level: {level}")

print("\n📅 DAY 1")

print(day1[[
    "Title",
    "stimulus",
    "Equipment",
    "Rating",
    "sets",
    "reps",
    "start_weight"
]])

print("\n📅 DAY 2")

print(day2[[
    "Title",
    "stimulus",
    "Equipment",
    "Rating",
    "sets",
    "reps",
    "start_weight"
]])


🟢 GLUTE AI — FINAL DETERMINISTIC PROGRAM
Goal: bulking
Level: Beginner

📅 DAY 1
                                        Title    stimulus  Equipment  Rating  \
751                           hm monster walk  activation      bands     4.6   
706                     glute bridge step-out     tension  body only     5.1   
717    fyr2 glute bridge dumbbell floor press     tension   dumbbell     5.0   
779                   30 legs glute-ham raise     stretch    machine     4.3   
760  hm alternating slow step-down with chair   stability  body only     4.3   
669                        hip lift with band   metabolic      bands     6.0   

     sets   reps  start_weight  
751     3  12-20           5.0  
706     4   6-10           0.0  
717     4   6-10          12.6  
779     3   8-12          13.9  
760     3  10-15           0.0  
669     3  15-25           6.5  

📅 DAY 2
                                                 Title    stimulus  \
748                                 holman donke

In [167]:
import pandas as pd

# LOAD DATA
df = pd.read_csv("exercise_FINAL_CLEAN.csv")

# CLEAN
df["BodyPart"] = df["BodyPart"].str.strip()
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

# FILTER ABDOMINALS ONLY
abs_df = df[df["BodyPart"] == "Core"].copy()

# DROP NULL RATINGS FOR ANALYSIS
rated = abs_df.dropna(subset=["Rating"])

# =========================
# BASIC STATS
# =========================
print("\n📊 Core STATS")
print(rated["Rating"].describe())

# =========================
# HIGH RATED (TOP 10)
# =========================
top_abs = rated.sort_values("Rating", ascending=False).head(10)

print("\n🔥 TOP Core (HIGH RATED)")
print(top_abs[["Title", "Type", "Equipment", "Rating"]])

# =========================
# LOW RATED (BOTTOM 10 > 0)
# =========================
low_abs = rated[rated["Rating"] > 0].sort_values("Rating", ascending=True).head(10)

print("\n🪶 LOW Core (LOW NON-ZERO)")
print(low_abs[["Title", "Type", "Equipment", "Rating"]])

# =========================
# ZERO RATED (IMPORTANT)
# =========================
zero_abs = rated[rated["Rating"] == 0]

print("\n⚠️ ZERO RATED EXERCISES")
print(zero_abs[["Title", "Type", "Equipment", "Rating"]].head(20))


📊 Core STATS
count    148.000000
mean       5.811486
std        3.885058
min        0.000000
25%        0.000000
50%        8.100000
75%        8.800000
max        9.500000
Name: Rating, dtype: float64

🔥 TOP Core (HIGH RATED)
                               Title      Type  Equipment  Rating
5                   weighted pull-up  strength      other     9.5
9                     landmine twist  strength      other     9.5
18  romanian deadlift with dumbbells  strength   dumbbell     9.4
25                       elbow plank  strength  body only     9.3
26                        bottoms up  strength  body only     9.3
27  standing cable low-to-high twist  strength      cable     9.3
29             suspended ab fall-out  strength  body only     9.3
33          dumbbell v-sit cross jab  strength   dumbbell     9.3
36             dumbbell spell caster  strength   dumbbell     9.3
45                           pullups  strength  body only     9.2

🪶 LOW Core (LOW NON-ZERO)
                   

In [192]:
import pandas as pd
import numpy as np

# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv("exercise_RATING_CLEAN.csv")

df["Title"] = df["Title"].str.lower().str.strip()
df["BodyPart"] = df["BodyPart"].str.lower().str.strip()
df["Equipment"] = df["Equipment"].str.lower().str.strip()

df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df["Rating"] = df["Rating"].fillna(df["Rating"].mean())


# =========================================================
# FILTER
# =========================================================
def filter_data(focus):
    data = df.copy()

    if focus == "abs":
        data = data[data["BodyPart"].isin(["abdominals", "core"])]
        data = data[data["Title"].str.contains(
            "crunch|plank|leg raise|hanging|jackknife|bicycle|v-up|rollout|carry|twist|mountain",
            case=False, na=False
        )]

    elif focus == "core":
        data = data[data["BodyPart"] == "core"]
        data = data[data["Title"].str.contains(
            "plank|dead bug|bird dog|pallof|rotation|carry|rollout|anti",
            case=False, na=False
        )]

    return data


# =========================================================
# LEVELS
# =========================================================
LEVEL_MULT = {
    "Beginner": 0.85,
    "Intermediate": 1.0,
    "Expert": 1.15
}


# =========================================================
# STIMULUS CLASSIFICATION
# =========================================================
def get_stimulus(title):
    t = title.lower()

    if any(x in t for x in ["crunch", "leg raise", "hanging", "rollout", "v-up", "jackknife"]):
        return "tension"

    if any(x in t for x in ["plank", "pallof", "carry"]):
        return "stability"

    if any(x in t for x in ["mountain", "bicycle", "flutter"]):
        return "metabolic"

    if any(x in t for x in ["dead bug", "bird dog"]):
        return "activation"

    return "other"


df["stimulus"] = df["Title"].apply(get_stimulus)


# =========================================================
# EQUIPMENT FACTOR
# =========================================================
def equipment_factor(eq):
    return {
        "barbell": 1.0,
        "machine": 0.9,
        "dumbbell": 0.75,
        "cable": 0.7,
        "kettlebells": 0.65,
        "medicine ball": 0.6,
        "bands": 0.45,
        "exercise ball": 0.5,
        "body only": 0.0
    }.get(eq, 0.5)


# =========================================================
# SCORE ENGINE
# =========================================================
def compute_score(row, goal, level):

    base = row["Rating"] / 10

    stim_bonus = {
        "tension": 2.6,
        "stability": 2.4,
        "activation": 1.9,
        "metabolic": 1.8,
        "other": 1.4
    }.get(row["stimulus"], 1.4)

    eq_factor = equipment_factor(row["Equipment"])

    level_mult = LEVEL_MULT.get(level, 1.0)

    if goal == "cutting":
        goal_bonus = 1.35 if row["stimulus"] in ["metabolic", "stability"] else 1.1

    elif goal == "bulking":
        goal_bonus = 1.45 if row["stimulus"] == "tension" else 1.1

    else:  # recomp
        goal_bonus = 1.25

    score = (
        base * 3.0 +
        stim_bonus * 1.8 +
        eq_factor * 2.2 +
        goal_bonus * 2.0
    ) * level_mult

    return round(score, 3)


# =========================================================
# NORMALIZE
# =========================================================
def normalize(scores):
    min_s, max_s = min(scores), max(scores)

    if max_s - min_s < 1e-6:
        return [7.0] * len(scores)

    return [
        3.5 + ((s - min_s) / (max_s - min_s)) * 6.5
        for s in scores
    ]


# =========================================================
# PRESCRIPTION
# =========================================================
def prescribe(stim, goal, level):

    if stim == "tension":
        reps = "8-12"
    elif stim == "stability":
        reps = "20-40 sec"
    elif stim == "activation":
        reps = "12-15"
    else:
        reps = "15-25"

    sets = 4 if goal == "bulking" and stim == "tension" else 3

    if level == "Beginner":
        sets = min(3, sets)

    return sets, reps


# =========================================================
# START WEIGHT (SAFE + REALISTIC)
# =========================================================
def start_weight(row, level):

    eq = row["Equipment"]
    rating = row["Rating"]

    if eq == "body only":
        return 0.0

    base = rating * 1.5 * equipment_factor(eq)

    level_mult = {
        "Beginner": 0.6,
        "Intermediate": 1.0,
        "Expert": 1.25
    }.get(level, 1.0)

    weight = base * level_mult

    # safety cap
    if level == "Beginner":
        weight = min(weight, 10)

    return round(weight, 1)


# =========================================================
# BUILD ONE DAY
# =========================================================
def build_day(data, goal, level, used):

    scores = data.apply(lambda r: compute_score(r, goal, level), axis=1)
    data = data.copy()
    data["score"] = normalize(scores)

    data = data.sort_values("score", ascending=False)

    structure = {
        "tension": 2,
        "stability": 2,
        "activation": 1,
        "metabolic": 1
    }

    day = []

    for stim, count in structure.items():

        pool = data[
            (data["stimulus"] == stim) &
            (~data["Title"].isin(used))
        ]

        selected = pool.head(count)

        for _, row in selected.iterrows():

            sets, reps = prescribe(row["stimulus"], goal, level)
            weight = start_weight(row, level)

            day.append({
                "Title": row["Title"].title(),
                "Stimulus": row["stimulus"].title(),
                "Equipment": row["Equipment"].title(),
                "Rating": round(row["Rating"], 2),
                "Score": round(row["score"], 2),
                "Sets": sets,
                "Reps": reps,
                "Start_Weight": weight
            })

        used.update(selected["Title"].tolist())

    return pd.DataFrame(day)


# =========================================================
# 3 DAY GENERATOR
# =========================================================
def generate_3_day(goal, level, focus):

    data = filter_data(focus)
    used = set()

    day1 = build_day(data, goal, level, used)
    day2 = build_day(data, goal, level, used)
    day3 = build_day(data, goal, level, used)

    return day1, day2, day3


# =========================================================
# RUN
# =========================================================
print("🟢 FIT AI v9 — 3 DAY CORE/ABS SYSTEM")

goal = input("Goal (cutting/bulking/recomp): ").lower()
level = input("Level (Beginner/Intermediate/Expert): ").title()
focus = input("Focus (core/abs): ").lower()

d1, d2, d3 = generate_3_day(goal, level, focus)

print("\n📅 DAY 1\n", d1)
print("\n📅 DAY 2\n", d2)
print("\n📅 DAY 3\n", d3)

🟢 FIT AI v9 — 3 DAY CORE/ABS SYSTEM

📅 DAY 1
                            Title    Stimulus      Equipment  Rating  Score  \
0  Barbell Ab Rollout - On Knees     Tension        Barbell    8.90  10.00   
1                Plank Leg Raise     Tension      Body Only    7.01   6.85   
2          30 Cable Pallof Press   Stability          Cable    8.73   8.78   
3                   Pallof Press   Stability          Cable    8.40   8.67   
4         Exercise Ball Bird Dog  Activation  Exercise Ball    7.01   6.67   

   Sets       Reps  Start_Weight  
0     3       8-12           8.0  
1     3       8-12           0.0  
2     3  20-40 sec           5.5  
3     3  20-40 sec           5.3  
4     3      12-15           3.2  

📅 DAY 2
                                                Title    Stimulus  Equipment  \
0                 Metaburn Side Plank Oblique Crunch     Tension  Body Only   
1  Holman Elbow Plank To Alternating Side Plank C...     Tension  Body Only   
2                Straight-Ar

In [190]:
import pandas as pd
import numpy as np

# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv("exercise_RATING.csv")

# Normalize text
df["Title"] = df["Title"].astype(str).str.lower().str.strip()
df["BodyPart"] = df["BodyPart"].astype(str).str.lower().str.strip()
df["Equipment"] = df["Equipment"].astype(str).str.lower().str.strip()

df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df["Rating"] = df["Rating"].fillna(df["Rating"].mean())

# =========================================================
# FIX MISSING / BAD EQUIPMENT
# =========================================================
df["Equipment"] = df["Equipment"].replace(["nan", "", "none"], np.nan)
df["Equipment"] = df["Equipment"].fillna("unknown")


# =========================================================
# RECLASSIFY "OTHER" USING TITLE KEYWORDS
# =========================================================
def reclassify_equipment(row):
    title = row["Title"]
    eq = row["Equipment"]

    # keep real equipment unchanged
    real_equipment = [
        "barbell", "dumbbell", "cable", "machine",
        "bands", "kettlebells", "medicine ball", "exercise ball"
    ]

    if eq in real_equipment or eq == "body only":
        return eq

    # -------------------------
    # OTHER → intelligent mapping
    # -------------------------
    if eq in ["other", "unknown"]:

        # HEAVY / LOADED CORE
        if any(x in title for x in [
            "rollout", "ab wheel", "pallof", "landmine",
            "carry", "press sit-up", "weighted"
        ]):
            return "cable"

        # DYNAMIC CORE / BODY CONTROL
        if any(x in title for x in [
            "plank", "dead bug", "bird dog",
            "mountain climber", "hollow", "jackknife"
        ]):
            return "body only"

        # FLEXION / CRUNCH MOVEMENTS
        if any(x in title for x in [
            "crunch", "sit-up", "leg raise", "v-up"
        ]):
            return "body only"

        # DEFAULT fallback
        return "body only"

    return eq


df["Equipment"] = df.apply(reclassify_equipment, axis=1)


# =========================================================
# VERIFY RESULTS
# =========================================================
print("\n📊 NEW EQUIPMENT DISTRIBUTION\n")
print(df["Equipment"].value_counts())

print("\n📊 CORE + ABS CHECK\n")
print(df[df["BodyPart"].isin(["core", "abdominals"])].groupby("Equipment").size())


# =========================================================
# SAVE CLEANED FILE
# =========================================================
output_file = "exercise_RATING_CLEAN.csv"
df.to_csv(output_file, index=False)

print(f"\n💾 Saved cleaned dataset: {output_file}")


📊 NEW EQUIPMENT DISTRIBUTION

Equipment
body only        1278
dumbbell          507
cable             277
barbell           277
machine           171
kettlebells       146
bands              97
medicine ball      38
exercise ball      35
plyometrics        26
e-z curl bar       21
foam roll          11
Name: count, dtype: int64

📊 CORE + ABS CHECK

Equipment
bands             23
barbell           21
body only        421
cable             61
dumbbell          72
e-z curl bar       1
exercise ball     18
kettlebells       28
machine            7
medicine ball     21
dtype: int64

💾 Saved cleaned dataset: exercise_RATING_CLEAN.csv
